# Active Preference Learning via Polytope Volume Removal

Instead of heuristic sample filtering, we maintain the feasible set $\Omega_t$ as an **explicit polytope** $\{\omega : A\omega \leq b\}$.

Each pairwise response (left, right, indifferent, incomparable) adds **linear constraints** on $\omega$,
and we select queries to **maximize expected volume removal** (Sadigh et al. 2017).

### Constraint rules (from the frame model)

Given query gaps $\Delta_j$ and thresholds $\tau, \tau'$:

| Response | Constraints on $\omega$ |
|----------|------------------------|
| **Left** ($Y \succ Y'$) | $\sum \omega_j |\Delta_j| \geq \tau$ and $\sum \omega_j (\Delta_j - \tau' |\Delta_j|) \geq 0$ |
| **Right** ($Y \prec Y'$) | $\sum \omega_j |\Delta_j| \geq \tau$ and $\sum \omega_j (\Delta_j + \tau' |\Delta_j|) \leq 0$ |
| **Indifferent** ($Y \sim Y'$) | $\sum \omega_j |\Delta_j| \leq \tau - \eta$ |
| **Incomparable** ($Y \bowtie Y'$) | $\sum \omega_j |\Delta_j| \geq \tau$, $\sum \omega_j (\Delta_j - \tau' |\Delta_j|) < 0$, $\sum \omega_j (\Delta_j + \tau' |\Delta_j|) > 0$ |

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog
from scipy.spatial.distance import pdist
from typing import List, Tuple, Optional, Set, Callable
from dataclasses import dataclass
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

# ============================================================================
# Configuration
# ============================================================================

FEATURE_NAMES = ['elderlyDep', 'lifeYearsGained', 'obesity', 'weeklyWorkhours', 'yearsWaiting']

FEATURE_RANGES = {
    'elderlyDep': (0, 5),
    'lifeYearsGained': (0, 25),
    'obesity': (0, 5),
    'weeklyWorkhours': (0, 50),
    'yearsWaiting': (1, 10)
}

# Algorithm parameters (can be changed at test time)
TAU = 0.1           # Intensity threshold
TAU_PRIME = 0.1     # Resolvability threshold
LAMBDA_X = 1.0      # Query scaling factor

print(f'Features: {FEATURE_NAMES}')
print(f'tau={TAU}, tau_prime={TAU_PRIME}, lambda_x={LAMBDA_X}')


# ============================================================================
# Data structures
# ============================================================================

@dataclass
class Patient:
    """Represents a patient with feature values."""
    elderlyDep: float
    lifeYearsGained: float
    obesity: float
    weeklyWorkhours: float
    yearsWaiting: float

    def to_array(self) -> np.ndarray:
        """Convert to numpy array in standard feature order."""
        return np.array([
            self.elderlyDep,
            self.lifeYearsGained,
            self.obesity,
            self.weeklyWorkhours,
            self.yearsWaiting
        ], dtype=float)

    @classmethod
    def from_array(cls, arr: np.ndarray) -> 'Patient':
        """Create Patient from numpy array."""
        return cls(
            elderlyDep=float(arr[0]),
            lifeYearsGained=float(arr[1]),
            obesity=float(arr[2]),
            weeklyWorkhours=float(arr[3]),
            yearsWaiting=float(arr[4])
        )

    def __repr__(self):
        return f"Patient(elder={self.elderlyDep}, life={self.lifeYearsGained}, " \
               f"obesity={self.obesity}, work={self.weeklyWorkhours}, wait={self.yearsWaiting})"


@dataclass
class PairwiseQuery:
    """Represents a pairwise comparison query."""
    patient_left: Patient
    patient_right: Patient
    context: Optional[str] = None

    def __repr__(self):
        return f"Query:\n  LEFT:  {self.patient_left}\n  RIGHT: {self.patient_right}"


# ============================================================================
# Query generation
# ============================================================================

def generate_random_patient() -> Patient:
    """Generate a random patient with features in valid ranges."""
    return Patient(
        elderlyDep=np.random.randint(0, 5),
        lifeYearsGained=np.random.randint(0, 25),
        obesity=np.random.randint(0, 5),
        weeklyWorkhours=np.random.randint(0, 50),
        yearsWaiting=np.random.randint(1, 10)
    )


def generate_random_patient_normalized() -> Patient:
    """Generate a random patient with features normalized to [0, 1]."""
    return Patient(
        elderlyDep=np.random.uniform(0, 1),
        lifeYearsGained=np.random.uniform(0, 1),
        obesity=np.random.uniform(0, 1),
        weeklyWorkhours=np.random.uniform(0, 1),
        yearsWaiting=np.random.uniform(0, 1)
    )


def generate_candidate_queries_normalized(n_candidates: int = 50) -> List[PairwiseQuery]:
    """Generate candidate queries with features normalized to [0,1]."""
    candidates = []
    for _ in range(n_candidates):
        left = generate_random_patient_normalized()
        right = generate_random_patient_normalized()
        candidates.append(PairwiseQuery(left, right))
    return candidates


print("Data structures defined: Patient, PairwiseQuery")
print("Query generators: generate_random_patient, generate_random_patient_normalized, generate_candidate_queries_normalized")


Features: ['elderlyDep', 'lifeYearsGained', 'obesity', 'weeklyWorkhours', 'yearsWaiting']
tau=0.1, tau_prime=0.1, lambda_x=1.0
Data structures defined: Patient, PairwiseQuery
Query generators: generate_random_patient, generate_random_patient_normalized, generate_candidate_queries_normalized


In [2]:
# ============================================================================
# Core Frame Model Computations
# ============================================================================

def compute_frame_gaps(
    query: PairwiseQuery,
    lambda_x: float = LAMBDA_X,
    V: np.ndarray = None,
    tau: float = TAU
) -> Tuple[np.ndarray, Set[int]]:
    """
    Compute frame-level gaps and identify active (decisive) frames.

    Parameters
    ----------
    query : PairwiseQuery
        The comparison query
    lambda_x : float
        Scaling factor for feature differences
    V : np.ndarray, optional
        Sign matrix (diagonal) to handle negative weights. 
        V[i,i] = -1 if oracle weight i is negative, else +1.
    tau : float
        Threshold for determining active frames

    Returns
    -------
    gaps : np.ndarray, shape (n_frames,)
        Gap for each frame: lambda_x * V @ (left_j - right_j)
    active_frames : Set[int]
        Frames where |gap_j| > tau (frames that "speak" to this query)
    """
    left_features = query.patient_left.to_array()
    right_features = query.patient_right.to_array()
    feature_diff = left_features - right_features
    if V is not None:
        gaps = lambda_x * (V @ feature_diff)
    else:
        gaps = lambda_x * feature_diff
    active_frames = set(np.where(np.abs(gaps) > 0)[0].tolist())
    return gaps, active_frames


def compute_aggregate_scores(
    gaps: np.ndarray,
    weights: np.ndarray,
    active_frames: Set[int]
) -> Tuple[float, float]:
    """
    Compute aggregate preference score delta(omega) and intensity r(omega).

    Parameters
    ----------
    gaps : np.ndarray
        Per-frame gaps from compute_frame_gaps
    weights : np.ndarray
        Weight vector omega (on simplex)
    active_frames : Set[int]
        Which frames are active for this query

    Returns
    -------
    delta_omega : float
        Weighted sum of gaps: sum_j omega_j * gap_j
        Positive = left is better, Negative = right is better
    r_omega : float
        Weighted sum of absolute gaps: sum_j omega_j * |gap_j|
        Measures "intensity" - how strongly the frames speak
    """
    if len(active_frames) == 0:
        return 0.0, 0.0

    active_list = sorted(list(active_frames))
    active_gaps = gaps[active_list]
    active_weights = weights[active_list]

    delta_omega = np.dot(active_weights, active_gaps)
    r_omega = np.dot(active_weights, np.abs(active_gaps))

    return delta_omega, r_omega


def predict_response(
    query: PairwiseQuery,
    weights: np.ndarray,
    tau: float = TAU,
    lambda_x: float = LAMBDA_X,
    tau_prime: float = TAU_PRIME,
    V: np.ndarray = None
) -> str:
    """
    Predict response for a query given a weight vector (DETERMINISTIC).

    Parameters
    ----------
    query : PairwiseQuery
        The comparison query
    weights : np.ndarray
        Weight vector omega (on simplex, all non-negative)
    tau : float
        Intensity threshold
    lambda_x : float
        Scaling factor
    tau_prime : float
        Resolvability threshold
    V : np.ndarray, optional
        Sign matrix (diagonal) to handle negative oracle weights.

    Decision rule:
    - r < tau           -> 'indifferent'  (not enough intensity)
    - |delta| < tau'*r  -> 'incomparable' (frames disagree)
    - delta >= tau'*r   -> 'left'         (left is better)
    - delta <= -tau'*r  -> 'right'        (right is better)

    Returns one of: 'left', 'right', 'indifferent', 'incomparable'
    """
    gaps, active_frames = compute_frame_gaps(query, lambda_x=lambda_x, V=V, tau=tau)
    delta_omega, r_omega = compute_aggregate_scores(gaps, weights, active_frames)

    if r_omega < tau:
        return 'indifferent'
    if r_omega >= tau and np.abs(delta_omega) < tau_prime * r_omega:
        return 'incomparable'
    elif r_omega >= tau and delta_omega >= tau_prime * r_omega:
        return 'left'
    elif r_omega >= tau and delta_omega <= -tau_prime * r_omega:
        return 'right'
    else:
        return 'indifferent'


def make_sign_matrix(beta: np.ndarray) -> np.ndarray:
    """
    Create diagonal sign matrix V from weight vector beta.
    
    V[i,i] = -1 if beta[i] < 0, else +1
    
    This allows the polytope algorithm (which requires non-negative weights)
    to handle negative oracle weights by absorbing signs into V.
    
    To recover signed weights: beta = V @ omega (where omega >= 0)
    """
    return np.diag(np.sign(beta))


print("Core functions defined:")
print("  - compute_frame_gaps(query, lambda_x, V, tau) -> (gaps, active_frames)")
print("  - compute_aggregate_scores(gaps, weights, active_frames) -> (delta_omega, r_omega)")
print("  - predict_response(query, weights, tau, lambda_x, tau_prime, V) -> response")
print("  - make_sign_matrix(beta) -> V (diagonal sign matrix)")

Core functions defined:
  - compute_frame_gaps(query, lambda_x, V, tau) -> (gaps, active_frames)
  - compute_aggregate_scores(gaps, weights, active_frames) -> (delta_omega, r_omega)
  - predict_response(query, weights, tau, lambda_x, tau_prime, V) -> response
  - make_sign_matrix(beta) -> V (diagonal sign matrix)


In [3]:
# ============================================================================
# Noise Injection for Latent Margins (Δ, r) - Unified Noise Model
# ============================================================================
#
# We add noise to latent margins:
#   Δ̃ = Δ + ε_Δ   (additive noise on preference score)
#   r̃ = r * exp(ε_r)  (multiplicative noise on intensity, keeps r̃ > 0)
#
# When scale_r = 0, only Δ gets noise (BT-equivalent model).
# When scale_r > 0, both Δ and r get noise (2D model).
#
# The decision rule uses (Δ̃, r̃):
#   - indifferent if r̃ < τ
#   - incomparable if r̃ ≥ τ and |Δ̃| < τ'r̃
#   - left if r̃ ≥ τ and Δ̃ ≥ τ'r̃
#   - right if r̃ ≥ τ and Δ̃ ≤ -τ'r̃
# ============================================================================

from typing import Callable, Tuple


def no_noise(delta_omega: float, r_omega: float) -> Tuple[float, float]:
    """No noise - deterministic model."""
    return delta_omega, r_omega


def create_noise_fn(
    noise_type: str = 'logistic',
    scale_delta: float = 1.0,
    scale_r: float = 0.0
) -> Callable:
    """
    Create a noise function for the latent margin model.
    
    Args:
        noise_type: 'logistic' or 'normal'
        scale_delta: Scale for Δ noise (additive)
        scale_r: Scale for r noise (multiplicative via exp). 0 = no r noise.
    
    Returns:
        noise_fn(delta, r) -> (delta_tilde, r_tilde)
    
    When scale_r = 0, this is equivalent to the BT model (delta-only noise).
    When scale_r > 0, r gets multiplicative noise: r̃ = r * exp(ε_r), keeping r̃ > 0.
    """
    def noise_fn(delta_omega: float, r_omega: float) -> Tuple[float, float]:
        # Delta noise (always applied, additive)
        if noise_type == 'logistic':
            eps_delta = np.random.logistic(loc=0, scale=scale_delta)
        else:  # normal
            eps_delta = np.random.normal(loc=0, scale=scale_delta)
        
        delta_tilde = delta_omega + eps_delta
        
        # r noise (multiplicative to keep r > 0)
        if scale_r > 0:
            if noise_type == 'logistic':
                eps_r = np.random.logistic(loc=0, scale=scale_r)
            else:
                eps_r = np.random.normal(loc=0, scale=scale_r)
            r_tilde = r_omega * np.exp(eps_r)
        else:
            r_tilde = r_omega
        
        return delta_tilde, r_tilde
    
    return noise_fn


def predict_response_noisy(
    query: PairwiseQuery,
    weights: np.ndarray,
    noise_fn: Callable = no_noise,
    tau: float = TAU,
    lambda_x: float = LAMBDA_X,
    tau_prime: float = TAU_PRIME,
    V: np.ndarray = None,
) -> str:
    """
    Predict response with noise injection into latent margins (Δ, r).
    
    This function:
    1. Computes deterministic (Δ, r)
    2. Applies noise_fn to get (Δ̃, r̃)
    3. Applies decision rule using (Δ̃, r̃, τ, τ')
    """
    gaps, active_frames = compute_frame_gaps(query, lambda_x, V=V, tau=tau)
    delta, r = compute_aggregate_scores(gaps, weights, active_frames)
    
    # Inject noise
    delta_tilde, r_tilde = noise_fn(delta, r)
    
    # Decision rule with noisy values
    if r_tilde < tau:
        return 'indifferent'
    if np.abs(delta_tilde) < tau_prime * r_tilde:
        return 'incomparable'
    elif delta_tilde >= tau_prime * r_tilde:
        return 'left'
    else:
        return 'right'


print("Noise model unified:")
print("  - create_noise_fn(noise_type, scale_delta, scale_r): creates noise function")
print("    - scale_r=0: delta-only noise (BT-equivalent)")
print("    - scale_r>0: 2D noise model (multiplicative on r)")
print("  - predict_response_noisy: applies noise and returns response")


Noise model unified:
  - create_noise_fn(noise_type, scale_delta, scale_r): creates noise function
    - scale_r=0: delta-only noise (BT-equivalent)
    - scale_r>0: 2D noise model (multiplicative on r)
  - predict_response_noisy: applies noise and returns response


In [4]:
# ============================================================================
# Test: Verify BT Equivalence
# ============================================================================
# When tau=tau'=0 with logistic noise, predict_response_noisy should give
# the same probability distribution as Bradley-Terry.

from collections import Counter
from scipy.special import expit  # sigmoid function

np.random.seed(42)

# Create a simple test query
left = Patient(elderlyDep=0.8, lifeYearsGained=0.6, obesity=0.3, weeklyWorkhours=0.5, yearsWaiting=0.4)
right = Patient(elderlyDep=0.2, lifeYearsGained=0.4, obesity=0.7, weeklyWorkhours=0.5, yearsWaiting=0.6)
query = PairwiseQuery(left, right)

# Weight vector
weights = np.array([0.1, 0.5, 0.1, 0.1, 0.2])

# Compute delta_omega (what BT uses)
delta_x = left.to_array() - right.to_array()
delta_omega = np.dot(weights, delta_x)

print("Test Setup:")
print(f"  delta_x (left - right): {delta_x}")
print(f"  weights: {weights}")
print(f"  delta_omega = w . delta_x = {delta_omega:.4f}")
print()

# BT theoretical probability
bt_prob_left = expit(delta_omega)  # sigmoid(delta_omega)
print(f"Bradley-Terry P(left) = sigmoid({delta_omega:.4f}) = {bt_prob_left:.4f}")
print()

# Empirical test: run many trials with tau=tau'=0 and logistic noise
n_trials = 10000
responses = []
for _ in range(n_trials):
    r = predict_response_noisy(
        query, weights, 
        noise_fn=create_noise_fn('logistic', scale_delta=1.0),
        tau=0.0, tau_prime=0.0, lambda_x=1.0, V=None
    )
    responses.append(r)

counts = Counter(responses)
empirical_prob_left = counts['left'] / n_trials

print(f"Empirical test ({n_trials} trials, tau=tau'=0, logistic noise):")
print(f"  Response counts: {dict(counts)}")
print(f"  Empirical P(left) = {empirical_prob_left:.4f}")
print()
print(f"Match: BT={bt_prob_left:.4f} vs Empirical={empirical_prob_left:.4f}")
print(f"  Difference: {abs(bt_prob_left - empirical_prob_left):.4f}")
if abs(bt_prob_left - empirical_prob_left) < 0.02:
    print("  ✓ Close match! Frame model with logistic noise ≈ Bradley-Terry")
else:
    print("  ✗ Mismatch - something is wrong")


Test Setup:
  delta_x (left - right): [ 0.6  0.2 -0.4  0.  -0.2]
  weights: [0.1 0.5 0.1 0.1 0.2]
  delta_omega = w . delta_x = 0.0800

Bradley-Terry P(left) = sigmoid(0.0800) = 0.5200

Empirical test (10000 trials, tau=tau'=0, logistic noise):
  Response counts: {'right': 4864, 'left': 5136}
  Empirical P(left) = 0.5136

Match: BT=0.5200 vs Empirical=0.5136
  Difference: 0.0064
  ✓ Close match! Frame model with logistic noise ≈ Bradley-Terry


In [5]:
# ============================================================================
# Likelihood Computation with Monte Carlo (for 2D noise)
# ============================================================================
#
# With noise on both Δ and r, we use Monte Carlo to estimate:
#   P(y | q, ω) = E_{ε_Δ, ε_r}[1{rule(Δ + ε_Δ, r * exp(ε_r)) = y}]
#
# We sample K noise pairs, apply the decision rule, and estimate
# probabilities by frequencies.
# ============================================================================

from scipy.special import expit as sigmoid
from scipy.stats import logistic as logistic_dist, norm as normal_dist


def apply_decision_rule(delta_tilde: float, r_tilde: float, 
                        tau: float, tau_prime: float) -> str:
    """Apply the frame model decision rule to noisy margins."""
    if r_tilde < tau:
        return 'indifferent'
    if np.abs(delta_tilde) < tau_prime * r_tilde:
        return 'incomparable'
    elif delta_tilde >= tau_prime * r_tilde:
        return 'left'
    else:
        return 'right'


def compute_response_probs_mc(
    query: PairwiseQuery,
    omega: np.ndarray,
    noise_type: str,
    scale_delta: float,
    scale_r: float,
    tau: float,
    tau_prime: float,
    lambda_x: float,
    V: np.ndarray = None,
    n_mc_samples: int = 500,
) -> np.ndarray:
    """
    Compute P(y | q, ω) for all 4 response types using Monte Carlo.
    
    Returns: probs array [p_left, p_right, p_indifferent, p_incomparable]
    """
    # Compute deterministic margins
    gaps, active_frames = compute_frame_gaps(query, lambda_x, V=V, tau=tau)
    delta, r = compute_aggregate_scores(gaps, omega, active_frames)
    
    # Count responses over MC samples
    counts = {'left': 0, 'right': 0, 'indifferent': 0, 'incomparable': 0}
    
    for _ in range(n_mc_samples):
        # Sample noise
        if noise_type == 'logistic':
            eps_delta = np.random.logistic(0, scale_delta)
            eta_r = np.random.logistic(0, scale_r)
        elif noise_type == 'normal':
            eps_delta = np.random.normal(0, scale_delta)
            eta_r = np.random.normal(0, scale_r)
        else:
            eps_delta = 0
            eta_r = 0
        
        # Apply noise (additive on Δ, multiplicative on r)
        delta_tilde = delta + eps_delta
        r_tilde = r * np.exp(eta_r) if scale_r > 0 else r
        
        # Get response
        response = apply_decision_rule(delta_tilde, r_tilde, tau, tau_prime)
        counts[response] += 1
    
    # Convert to probabilities
    probs = np.array([
        counts['left'] / n_mc_samples,
        counts['right'] / n_mc_samples,
        counts['indifferent'] / n_mc_samples,
        counts['incomparable'] / n_mc_samples,
    ])
    
    return probs


def compute_response_log_likelihood(
    query: PairwiseQuery,
    response: str,
    omega: np.ndarray,
    noise_type: str = 'logistic',
    scale_delta: float = 1.0,
    scale_r: float = 0.0,  # Default 0 = no r noise (backwards compatible)
    tau: float = TAU,
    tau_prime: float = TAU_PRIME,
    lambda_x: float = LAMBDA_X,
    V: np.ndarray = None,
    n_mc_samples: int = 500,
) -> float:
    """
    Compute log P(response | query, omega) using Monte Carlo.
    
    When scale_r = 0, this reduces to the original 1D noise model
    (though still uses MC instead of closed-form).
    """
    # Deterministic case
    if noise_type == 'none' or (scale_delta == 0 and scale_r == 0):
        predicted = predict_response(query, omega, tau, lambda_x, tau_prime, V=V)
        return 0.0 if response == predicted else -np.inf
    
    # Monte Carlo estimation
    probs = compute_response_probs_mc(
        query, omega, noise_type, scale_delta, scale_r,
        tau, tau_prime, lambda_x, V, n_mc_samples
    )
    
    response_idx = {'left': 0, 'right': 1, 'indifferent': 2, 'incomparable': 3}
    prob = probs[response_idx[response]]
    
    return np.log(max(prob, 1e-10))


def compute_transcript_log_likelihood(
    transcript: List[Tuple[PairwiseQuery, str]],
    omega: np.ndarray,
    noise_type: str = 'logistic',
    scale_delta: float = 1.0,
    scale_r: float = 0.0,
    tau: float = TAU,
    tau_prime: float = TAU_PRIME,
    lambda_x: float = LAMBDA_X,
    V: np.ndarray = None,
    n_mc_samples: int = 200,  # Fewer samples per query for speed
) -> float:
    """Compute total log-likelihood of a transcript."""
    ll = 0.0
    for query, response in transcript:
        ll += compute_response_log_likelihood(
            query, response, omega, noise_type, scale_delta, scale_r,
            tau, tau_prime, lambda_x, V, n_mc_samples
        )
    return ll


# Also provide closed-form version for delta-only noise (faster)
def compute_response_probs_closed_form(
    query: PairwiseQuery,
    omega: np.ndarray,
    noise_type: str,
    scale: float,
    tau: float,
    tau_prime: float,
    lambda_x: float,
    V: np.ndarray = None,
) -> np.ndarray:
    """
    Closed-form P(y|q,ω) when noise is only on Δ (not r).
    This is the original fast version.
    """
    gaps, active_frames = compute_frame_gaps(query, lambda_x, V=V, tau=tau)
    delta, r = compute_aggregate_scores(gaps, omega, active_frames)
    
    probs = np.zeros(4)
    
    if r < tau:
        probs[2] = 1.0  # indifferent
        return probs
    
    # Thresholds for noise variable
    thr_L = tau_prime * r - delta
    thr_R = -tau_prime * r - delta
    
    if noise_type == 'logistic':
        sL = sigmoid(thr_L / scale)
        sR = sigmoid(thr_R / scale)
        probs[0] = 1.0 - sL         # left
        probs[1] = sR               # right
        probs[3] = sL - sR          # incomparable
    elif noise_type == 'normal':
        probs[0] = 1.0 - normal_dist.cdf(thr_L / scale)
        probs[1] = normal_dist.cdf(thr_R / scale)
        probs[3] = normal_dist.cdf(thr_L / scale) - normal_dist.cdf(thr_R / scale)
    
    probs = np.clip(probs, 0.0, 1.0)
    probs /= probs.sum() + 1e-15
    return probs


print("Likelihood functions defined:")
print("  - compute_response_probs_mc: MC estimation for 2D noise")
print("  - compute_response_probs_closed_form: fast closed-form for Δ-only noise")
print("  - compute_response_log_likelihood: uses MC (set scale_r=0 for Δ-only)")
print("  - compute_transcript_log_likelihood: sum of log likelihoods")


Likelihood functions defined:
  - compute_response_probs_mc: MC estimation for 2D noise
  - compute_response_probs_closed_form: fast closed-form for Δ-only noise
  - compute_response_log_likelihood: uses MC (set scale_r=0 for Δ-only)
  - compute_transcript_log_likelihood: sum of log likelihoods


In [6]:
# ============================================================================
# Bayesian Inference via Hit-and-Run MCMC with Metropolis-Hastings
# ============================================================================

def hit_and_run_simplex_step(x: np.ndarray) -> np.ndarray:
    """
    One hit-and-run step on the simplex {w : sum(w)=1, w>=0}.
    """
    dim = len(x)
    
    # Random direction projected onto sum=0 hyperplane
    d = np.random.randn(dim)
    d = d - d.mean()
    norm = np.linalg.norm(d)
    if norm < 1e-12:
        return x.copy()
    d = d / norm
    
    # Find t bounds
    t_min, t_max = -np.inf, np.inf
    for j in range(dim):
        if d[j] > 1e-12:
            t_min = max(t_min, -x[j] / d[j])
        elif d[j] < -1e-12:
            t_max = min(t_max, -x[j] / d[j])
    
    if t_min >= t_max - 1e-12:
        return x.copy()
    
    t = np.random.uniform(t_min, t_max)
    new_x = x + t * d
    
    if new_x.min() < -1e-10 or abs(new_x.sum() - 1.0) > 1e-10:
        return x.copy()
    
    new_x = np.maximum(new_x, 0.0)
    new_x = new_x / new_x.sum()
    return new_x


def sample_posterior_hit_and_run(
    transcript: List[Tuple[PairwiseQuery, str]],
    noise_type: str = 'logistic',
    scale_delta: float = 1.0,
    scale_r: float = 0.0,  # 0 = no r noise
    tau: float = TAU,
    tau_prime: float = TAU_PRIME,
    lambda_x: float = LAMBDA_X,
    n_samples: int = 2000,
    burn_in: int = 1000,
    thin: int = 1,
    n_mc_samples: int = 100,  # MC samples for likelihood
    verbose: bool = True,
    V: np.ndarray = None,
) -> Tuple[np.ndarray, float]:
    """
    Sample from posterior P(omega | transcript) using hit-and-run + MH.
    
    Uses Monte Carlo likelihood estimation for 2D noise model.
    """
    dim = len(FEATURE_NAMES)
    
    # Initialize at simplex center
    omega = np.ones(dim) / dim
    ll_current = compute_transcript_log_likelihood(
        transcript, omega, noise_type, scale_delta, scale_r,
        tau, tau_prime, lambda_x, V, n_mc_samples
    )
    
    samples = []
    n_accepted = 0
    total_steps = burn_in + n_samples * thin
    
    for step in range(total_steps):
        # Hit-and-run proposal
        proposal = hit_and_run_simplex_step(omega)
        
        # Compute proposal likelihood
        ll_proposal = compute_transcript_log_likelihood(
            transcript, proposal, noise_type, scale_delta, scale_r,
            tau, tau_prime, lambda_x, V, n_mc_samples
        )
        
        # MH acceptance
        log_alpha = ll_proposal - ll_current
        
        if np.log(np.random.rand()) < log_alpha:
            omega = proposal
            ll_current = ll_proposal
            if step >= burn_in:
                n_accepted += 1
        
        if step >= burn_in and (step - burn_in) % thin == 0:
            samples.append(omega.copy())
    
    acceptance_rate = n_accepted / max(1, n_samples * thin)
    if verbose:
        print(f"MCMC: {len(samples)} samples, acceptance = {acceptance_rate:.1%}, "
              f"final LL = {ll_current:.2f}")
    
    return np.array(samples), acceptance_rate


print("MCMC sampler updated for 2D noise model")
print("  - sample_posterior_hit_and_run now takes scale_delta and scale_r")
print("  - Uses Monte Carlo likelihood estimation")


MCMC sampler updated for 2D noise model
  - sample_posterior_hit_and_run now takes scale_delta and scale_r
  - Uses Monte Carlo likelihood estimation


In [7]:
# ============================================================================
# Active Learning with BALD Query Selection (2D Noise Model)
# ============================================================================

def compute_response_probs(
    query: PairwiseQuery,
    omega: np.ndarray,
    noise_type: str,
    scale_delta: float,
    scale_r: float,
    tau: float,
    tau_prime: float,
    lambda_x: float,
    V: np.ndarray = None,
    n_mc_samples: int = 100,
) -> np.ndarray:
    """
    Compute P(y | q, ω) for all 4 responses.
    Uses closed-form if scale_r=0, otherwise MC.
    """
    if scale_r == 0 or scale_r is None:
        # Use fast closed-form for delta-only noise
        return compute_response_probs_closed_form(
            query, omega, noise_type, scale_delta, tau, tau_prime, lambda_x, V
        )
    else:
        # Use MC for 2D noise
        return compute_response_probs_mc(
            query, omega, noise_type, scale_delta, scale_r,
            tau, tau_prime, lambda_x, V, n_mc_samples
        )


def entropy(probs: np.ndarray) -> float:
    """Compute entropy H[p] = -sum(p * log(p))"""
    probs = np.clip(probs, 1e-15, 1.0)
    return -np.sum(probs * np.log(probs))


from typing import Literal

AcqObjective = Literal["full4", "bt_per_decisive", "bt_per_attempt"]

def bald_score(
    query: PairwiseQuery,
    posterior_samples: np.ndarray,
    noise_type: str,
    scale_delta: float,
    scale_r: float,
    tau: float,
    tau_prime: float,
    lambda_x: float,
    max_samples: int = 100,
    V: np.ndarray = None,
    n_mc_samples: int = 50,
    objective: AcqObjective = "full4",
) -> float:
    """
    Compute acquisition score for a query using BALD-style mutual information.

    Objectives:
      - "full4":
          Standard BALD on Y ∈ {L, R, I, Inc}.
          score = I(ω; Y | q, D)

      - "bt_per_decisive":
          Conditional BALD on decisive outcomes only (L/R),
          i.e. score = I(ω; Y | q, D=1), where D=1 iff Y ∈ {L, R}.
          This is principled when your *budget is decisive labels* (e.g., "give BT 200 usable").

      - "bt_per_attempt":
          Expected information gain per attempt for a learner that *updates only on L/R*.
          score = E[p_dec(q,ω)] * I(ω; Y | q, D=1)
          This is principled when your budget is *attempts/questions asked* and BT ignores I/Inc in updates.

    Notes:
      - Assumes compute_response_probs returns probs in order:
        [P(L), P(R), P(I), P(Inc)].
    """
    n_samples = min(len(posterior_samples), max_samples)

    if n_samples == 0:
        return 0.0

    # Collect per-sample predictive distributions depending on objective
    all_probs = []
    p_dec_list = []  # only used for bt_per_attempt

    for omega in posterior_samples[:n_samples]:
        p4 = compute_response_probs(
            query, omega, noise_type, scale_delta, scale_r,
            tau, tau_prime, lambda_x, V, n_mc_samples
        )

        # Make sure it's safe numerically
        p4 = np.clip(p4, 0.0, 1.0)
        p4 = p4 / (p4.sum() + 1e-15)

        if objective == "full4":
            probs = p4

        elif objective in ("bt_per_decisive", "bt_per_attempt"):
            p_dec = float(p4[0] + p4[1])
            if objective == "bt_per_attempt":
                p_dec_list.append(p_dec)

            if p_dec > 1e-12:
                probs = np.array([p4[0] / p_dec, p4[1] / p_dec], dtype=float)  # P(L|dec), P(R|dec)
            else:
                # If almost never decisive under this ω, conditional distribution is irrelevant
                # (and for bt_per_attempt it'll be downweighted by p_dec anyway)
                probs = np.array([0.5, 0.5], dtype=float)

        else:
            raise ValueError(f"Unknown objective: {objective}")

        all_probs.append(probs)

    all_probs = np.array(all_probs)
    avg_probs = all_probs.mean(axis=0)

    # BALD = H[E p] - E H[p]
    bald = entropy(avg_probs) - float(np.mean([entropy(p) for p in all_probs]))

    if objective == "bt_per_attempt":
        p_dec_mean = float(np.mean(p_dec_list)) if len(p_dec_list) else 0.0
        return p_dec_mean * bald

    return bald


def active_learning_bayesian(
    oracle_weights: np.ndarray,
    noise_type: str = 'logistic',
    scale_delta: float = 1.0,
    scale_r: float = 0.0,
    tau: float = TAU,
    tau_prime: float = TAU_PRIME,
    lambda_x: float = LAMBDA_X,
    max_iterations: int = 50,
    n_candidates: int = 100,
    n_posterior_samples: int = 1000,
    target_diameter: float = 0.1,
    n_mc_samples: int = 50,
    verbose: bool = True,
    V: np.ndarray = None,
    objective: AcqObjective = "full4",
) -> dict:
    """
    Active learning with BALD query selection.
    Supports 2D noise model (noise on both Δ and r).
    
    Args:
        objective: Query selection objective.
            - "full4": Standard BALD over all 4 outcomes {L, R, I, Inc}
            - "bt_per_decisive": Conditional BALD over {L, R} only (per usable label)
            - "bt_per_attempt": E[p_dec] * conditional BALD (per attempt, BT-style)
    """
    dim = len(FEATURE_NAMES)
    transcript = []
    history = []
    
    # Create noise function for oracle
    oracle_noise_fn = create_noise_fn(noise_type, scale_delta, scale_r)
    
    if verbose:
        print("=" * 60)
        print(f"Bayesian Active Learning with BALD ({objective})")
        if objective == "bt_per_decisive":
            print("Query selection: conditional BALD over {L,R} (per usable label)")
        elif objective == "bt_per_attempt":
            print("Query selection: E[p_dec] * conditional BALD (per attempt)")
        else:
            print("Query selection: full 4-way BALD over {L,R,I,Inc}")
        print("=" * 60)
        print(f"Oracle: {oracle_weights}")
        print(f"Noise: {noise_type}, scale_Δ={scale_delta}, scale_r={scale_r}")
        print(f"tau={tau}, tau'={tau_prime}")
        print()
    
    for iteration in range(max_iterations):
        # Get posterior samples
        if len(transcript) == 0:
            samples = np.random.dirichlet(np.ones(dim), size=n_posterior_samples)
        else:
            samples, acc_rate = sample_posterior_hit_and_run(
                transcript, noise_type, scale_delta, scale_r,
                tau, tau_prime, lambda_x,
                n_samples=n_posterior_samples, burn_in=500,
                n_mc_samples=n_mc_samples, verbose=False, V=V
            )
        
        # Compute metrics
        if len(samples) > 100:
            idx = np.random.choice(len(samples), 100, replace=False)
            diam = pdist(samples[idx], metric='cityblock').max()
        else:
            diam = pdist(samples, metric='cityblock').max() if len(samples) > 1 else 2.0
        
        posterior_mean = samples.mean(axis=0)
        cos_sim = np.dot(posterior_mean, oracle_weights) / (
            np.linalg.norm(posterior_mean) * np.linalg.norm(oracle_weights) + 1e-10
        )
        l1_err = np.abs(posterior_mean - oracle_weights).sum()
        
        if verbose:
            print(f"Iter {iteration+1:2d}: diam={diam:.3f}, cos={cos_sim:.3f}, L1={l1_err:.3f}")
        
        history.append({
            'iteration': iteration + 1,
            'diameter': diam,
            'cosine_similarity': cos_sim,
            'l1_error': l1_err,
            'posterior_mean': posterior_mean.copy(),
        })
        
        if diam <= target_diameter:
            if verbose:
                print("Converged!")
            break
        
        # Generate candidates and select using BALD
        candidates = generate_candidate_queries_normalized(n_candidates)
        
        best_query = None
        best_bald = -np.inf
        
        for query in candidates:
            score = bald_score(
                query, samples, noise_type, scale_delta, scale_r,
                tau, tau_prime, lambda_x, max_samples=100, V=V,
                n_mc_samples=n_mc_samples,
                objective=objective,
            )
            
            if score > best_bald:
                best_bald = score
                best_query = query
        
        if best_query is None:
            best_query = candidates[0]
        
        # Oracle response
        response = predict_response_noisy(
            best_query, oracle_weights, oracle_noise_fn,
            tau, lambda_x, tau_prime, V
        )
        transcript.append((best_query, response))
        
        if verbose and iteration < 5:
            print(f"       BALD={best_bald:.3f}, response={response}")
    
    # Final samples
    if len(transcript) > 0:
        final_samples, _ = sample_posterior_hit_and_run(
            transcript, noise_type, scale_delta, scale_r,
            tau, tau_prime, lambda_x,
            n_samples=n_posterior_samples, burn_in=500,
            n_mc_samples=n_mc_samples, verbose=False, V=V
        )
    else:
        final_samples = samples
    
    final_mean = final_samples.mean(axis=0)
    
    if verbose:
        print()
        print("=" * 60)
        print(f"Final: {len(transcript)} queries")
        print(f"Oracle:    {oracle_weights}")
        print(f"Estimate:  {final_mean}")
        print(f"L1 error:  {np.abs(final_mean - oracle_weights).sum():.4f}")
        # Count responses
        from collections import Counter
        resp_counts = Counter(r for _, r in transcript)
        print(f"Responses: {dict(resp_counts)}")
    
    return {
        'transcript': transcript,
        'posterior_mean': final_mean,
        'posterior_samples': final_samples,
        'history': history,
    }


print("Active learning with BALD query selection:")
print("  - bald_score: supports objective='full4', 'bt_per_decisive', 'bt_per_attempt'")
print("  - active_learning_bayesian: pass objective= to control query selection")

Active learning with BALD query selection:
  - bald_score: supports objective='full4', 'bt_per_decisive', 'bt_per_attempt'
  - active_learning_bayesian: pass objective= to control query selection


In [8]:
# ============================================================================
# TEST: Scale matching between BT and our model
# ============================================================================
# 
# Key insight: BT's implicit scale depends on the MAGNITUDE of weights.
# On the simplex (sum=1), our deltas are small, so we need smaller scale.
#
# Rule of thumb: If typical |delta| ≈ 0.3 on simplex, we need scale ≈ 0.3
# to get similar noise-to-signal ratio as BT with scale=1 and |delta| ≈ 1.
# ============================================================================

from scipy.optimize import minimize
import numpy as np

def bt_mle(
    transcript,
    dim: int,
    *,
    lambda_x: float = 1.0,
    scale: float = 1.0,
    l2_theta: float = 0.0,
    forced_choice: bool = False,
    seed: int | None = None,
    n_restarts: int = 10,
):
    """
    Bradley-Terry / logistic-regression MLE (or MAP if l2_theta>0) with a simplex-constrained
    weight vector w via a softmax parameterization.

    This is the apples-to-apples BT baseline for your tau=tau'=0 frame model when your
    likelihood is:
        P(left | q, w) = sigmoid( (lambda_x/scale) * (x_L - x_R)^T w )

    Parameters
    ----------
    transcript : list of (query, response) tuples
    dim : int
        Dimension of weight vector
    lambda_x : float
        Scaling factor for features
    scale : float
        Noise scale (matches frame model likelihood)
    l2_theta : float
        L2 regularization on theta (MAP if > 0, MLE if 0)
    forced_choice : bool
        If True, convert 'indifferent'/'incomparable' responses to random left/right.
        If False (default), skip non-decisive responses.
    seed : int or None
        Random seed for optimization restarts (and forced choice if enabled)
    n_restarts : int
        Number of random restarts for optimization

    Notes
    -----
    - By default, uses ONLY 'left'/'right' rows (BT is binary).
    - With forced_choice=True, 'indifferent'/'incomparable' become random left/right.
    - Constrains w to the simplex by w = softmax(theta).
    """
    rng = np.random.default_rng(seed)

    # Build design matrix on differences and binary labels
    X, y = [], []
    for query, response in transcript:
        if response in ("left", "right"):
            delta = query.patient_left.to_array() - query.patient_right.to_array()
            X.append(delta)
            y.append(1.0 if response == "left" else 0.0)
        elif forced_choice and response in ("indifferent", "incomparable"):
            # Force a random decision
            delta = query.patient_left.to_array() - query.patient_right.to_array()
            X.append(delta)
            y.append(1.0 if rng.random() < 0.5 else 0.0)  # coin flip

    if len(X) == 0:
        return np.ones(dim) / dim

    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    # Safe sigmoid
    def sigmoid(z):
        z = np.asarray(z)
        out = np.empty_like(z, dtype=float)
        pos = z >= 0
        out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
        ez = np.exp(z[~pos])
        out[~pos] = ez / (1.0 + ez)
        return out

    def softmax(theta):
        t = theta - np.max(theta)
        e = np.exp(t)
        return e / np.sum(e)

    # Negative log-likelihood in theta-space
    temp = lambda_x / scale

    def neg_log_post(theta):
        w = softmax(theta)
        logits = temp * (X @ w)
        p = sigmoid(logits)

        eps = 1e-12
        ll = np.sum(y * np.log(p + eps) + (1.0 - y) * np.log(1.0 - p + eps))

        reg = l2_theta * np.sum(theta * theta) if l2_theta > 0 else 0.0
        return -ll + reg

    best = None
    for _ in range(max(1, n_restarts)):
        theta0 = rng.normal(0.0, 0.1, size=dim)
        res = minimize(neg_log_post, theta0, method="L-BFGS-B")
        if best is None or res.fun < best.fun:
            best = res

    w_hat = softmax(best.x)
    return w_hat


print("bt_mle updated with forced_choice parameter:")
print("  - forced_choice=False (default): skip indifferent/incomparable")
print("  - forced_choice=True: convert to random left/right (coin flip)")


bt_mle updated with forced_choice parameter:
  - forced_choice=False (default): skip indifferent/incomparable
  - forced_choice=True: convert to random left/right (coin flip)


In [14]:
# ============================================================================
# EXPERIMENT: BT-optimized queries vs Full4 queries (both with Bayesian inference)
# ============================================================================
# 
# Question: If we optimize query selection for decisive outcomes (bt_per_attempt),
# does Bayesian inference suffer compared to using full4 query selection?
#
# Setup:
#   - tau = tau' = 0.25 (non-trivial indifference/incomparability regions)
#   - Fixed oracle weights
#   - Two conditions (BOTH use Bayesian posterior mean for inference):
#       1. BT-optimized queries: objective="bt_per_attempt"
#       2. Full4 queries: objective="full4"
#   - Query budgets: 10, 20, 30, ..., 100
#   - Metric: cosine similarity to true oracle
# ============================================================================

import numpy as np
from collections import Counter

# Experiment parameters
np.random.seed(42)
oracle_weights = np.random.dirichlet(np.ones(len(FEATURE_NAMES)))
print(f"Oracle weights: {oracle_weights}")

tau_exp = 0.25
tau_prime_exp = 0.25
scale_delta_exp = 0.3  # reasonable noise level
scale_r_exp = 0.0      # no r noise for simplicity
noise_type_exp = 'logistic'

query_budgets = [10, 20, 30, 40]

# Storage for results
results_bt_queries = []   # BT-optimized queries, Bayesian inference
results_full4_queries = []  # Full4 queries, Bayesian inference

print(f"\nExperiment settings:")
print(f"  tau = tau' = {tau_exp}")
print(f"  scale_delta = {scale_delta_exp}")
print(f"  Query budgets: {query_budgets}")
print(f"  Inference: Bayesian posterior mean (for both conditions)")
print()

# ============================================================================
# Run BT-optimized query selection (bt_per_attempt) with Bayesian inference
# ============================================================================
print("=" * 60)
print("Condition 1: BT-optimized queries (objective='bt_per_attempt')")
print("           + Bayesian inference (posterior mean)")
print("=" * 60)

for max_queries in query_budgets:
    result = active_learning_bayesian(
        oracle_weights,
        noise_type=noise_type_exp,
        scale_delta=scale_delta_exp,
        scale_r=scale_r_exp,
        tau=tau_exp,
        tau_prime=tau_prime_exp,
        lambda_x=LAMBDA_X,
        max_iterations=max_queries,
        n_candidates=100,
        n_posterior_samples=500,
        target_diameter=0.0,  # Don't stop early
        n_mc_samples=50,
        verbose=False,
        objective="bt_per_attempt",
    )
    
    transcript = result['transcript']
    posterior_mean = result['posterior_mean']
    
    # Count response types
    resp_counts = Counter(r for _, r in transcript)
    n_decisive = resp_counts.get('left', 0) + resp_counts.get('right', 0)
    
    # Cosine similarity using Bayesian posterior mean
    cos_sim = np.dot(posterior_mean, oracle_weights) / (
        np.linalg.norm(posterior_mean) * np.linalg.norm(oracle_weights) + 1e-10
    )
    
    results_bt_queries.append({
        'n_queries': max_queries,
        'n_decisive': n_decisive,
        'cos_sim': cos_sim,
        'estimate': posterior_mean,
        'responses': dict(resp_counts),
    })
    
    print(f"  {max_queries} queries: {n_decisive} decisive, cos_sim={cos_sim:.4f}")

print()

# ============================================================================
# Run Full4 query selection with Bayesian inference
# ============================================================================
print("=" * 60)
print("Condition 2: Full4 queries (objective='full4')")
print("           + Bayesian inference (posterior mean)")
print("=" * 60)

for max_queries in query_budgets:
    result = active_learning_bayesian(
        oracle_weights,
        noise_type=noise_type_exp,
        scale_delta=scale_delta_exp,
        scale_r=scale_r_exp,
        tau=tau_exp,
        tau_prime=tau_prime_exp,
        lambda_x=LAMBDA_X,
        max_iterations=max_queries,
        n_candidates=100,
        n_posterior_samples=500,
        target_diameter=0.0,  # Don't stop early
        n_mc_samples=50,
        verbose=False,
        objective="full4",
    )
    
    transcript = result['transcript']
    posterior_mean = result['posterior_mean']
    
    # Count response types
    resp_counts = Counter(r for _, r in transcript)
    n_decisive = resp_counts.get('left', 0) + resp_counts.get('right', 0)
    
    # Cosine similarity using Bayesian posterior mean
    cos_sim = np.dot(posterior_mean, oracle_weights) / (
        np.linalg.norm(posterior_mean) * np.linalg.norm(oracle_weights) + 1e-10
    )
    
    results_full4_queries.append({
        'n_queries': max_queries,
        'n_decisive': n_decisive,
        'cos_sim': cos_sim,
        'estimate': posterior_mean,
        'responses': dict(resp_counts),
    })
    
    print(f"  {max_queries} queries: {n_decisive} decisive, cos_sim={cos_sim:.4f}")

print()
print("Experiment complete!")

Oracle weights: [0.07982511 0.51203839 0.22398576 0.1552966  0.02885413]

Experiment settings:
  tau = tau' = 0.25
  scale_delta = 0.3
  Query budgets: [10, 20, 30, 40]
  Inference: Bayesian posterior mean (for both conditions)

Condition 1: BT-optimized queries (objective='bt_per_attempt')
           + Bayesian inference (posterior mean)
  10 queries: 7 decisive, cos_sim=0.8889
  20 queries: 16 decisive, cos_sim=0.9368
  30 queries: 18 decisive, cos_sim=0.9983
  40 queries: 36 decisive, cos_sim=0.9698

Condition 2: Full4 queries (objective='full4')
           + Bayesian inference (posterior mean)
  10 queries: 6 decisive, cos_sim=0.9870


KeyboardInterrupt: 

In [ ]:
# ============================================================================
# Plot: BT-optimized queries vs Full4 queries (both Bayesian inference)
# ============================================================================

import sys
sys.path.insert(0, '/Users/michellesi/Desktop/harvard/topmodel/learning-algo')
from style import paper_style, SINGLE_COLUMN, COLOR_A, COLOR_B

# Extract data for plotting
n_queries_bt = [r['n_queries'] for r in results_bt_queries]
cos_sim_bt = [r['cos_sim'] for r in results_bt_queries]
n_decisive_bt = [r['n_decisive'] for r in results_bt_queries]

n_queries_full4 = [r['n_queries'] for r in results_full4_queries]
cos_sim_full4 = [r['cos_sim'] for r in results_full4_queries]
n_decisive_full4 = [r['n_decisive'] for r in results_full4_queries]

with paper_style(width=SINGLE_COLUMN * 1.5, aspect=0.7, use_latex=False, font_size=9):
    fig, axes = plt.subplots(1, 2, figsize=(7, 3))
    
    # Left plot: Cosine similarity vs number of queries
    ax1 = axes[0]
    ax1.plot(n_queries_bt, cos_sim_bt, 'o-', color=COLOR_B, label='bt_per_attempt', markersize=6)
    ax1.plot(n_queries_full4, cos_sim_full4, 's-', color=COLOR_A, label='full4', markersize=6)
    ax1.set_xlabel('Number of Queries')
    ax1.set_ylabel('Cosine Similarity to Oracle')
    ax1.set_title(f'Bayesian Convergence (τ=τ\'={tau_exp})')
    ax1.legend(loc='lower right', title='Query Selection')
    ax1.set_ylim([0, 1.05])
    ax1.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)
    
    # Right plot: Number of decisive queries obtained
    ax2 = axes[1]
    ax2.plot(n_queries_bt, n_decisive_bt, 'o-', color=COLOR_B, label='bt_per_attempt', markersize=6)
    ax2.plot(n_queries_full4, n_decisive_full4, 's-', color=COLOR_A, label='full4', markersize=6)
    ax2.set_xlabel('Number of Queries')
    ax2.set_ylabel('Decisive Responses (L/R)')
    ax2.set_title('Query Type Distribution')
    ax2.legend(loc='lower right', title='Query Selection')
    # Add diagonal reference line (100% decisive)
    ax2.plot([0, 100], [0, 100], 'k--', alpha=0.3, linewidth=0.8, label='_nolegend_')
    
    plt.tight_layout()
    plt.savefig('query_selection_comparison.pdf', bbox_inches='tight')
    plt.show()

# Print summary statistics
print("\n" + "=" * 60)
print("Summary")
print("=" * 60)
print(f"\nAt 100 queries (both use Bayesian inference):")
print(f"  bt_per_attempt queries: cos_sim = {cos_sim_bt[-1]:.4f}, {n_decisive_bt[-1]} decisive")
print(f"  full4 queries:          cos_sim = {cos_sim_full4[-1]:.4f}, {n_decisive_full4[-1]} decisive")
print(f"\nResponse breakdown at 100 queries:")
print(f"  bt_per_attempt: {results_bt_queries[-1]['responses']}")
print(f"  full4:          {results_full4_queries[-1]['responses']}")